# 02 - Environment, reward, and per-action MC-return targets

Implements proposal section 5 end to end:

1. Volatility-dependent transaction cost `c_t = c_0 + lambda * sigma_t^(20)` (eq 21).
2. Per-step reward `R_t = a_t * r_{t+1} - c_t * |a_t - a_{t-1}|` (eq 22).
3. Monte Carlo return `y_t = sum_{k=0}^H gamma^k R_{t+k}` (eq 1).
4. Calibrate the prior scale `sigma_y_hat` on the pre-training window
   (^GSPC 1990-1992) under uniform random actions, then freeze it.
5. Show the per-action distribution of `y_t` on a SPY training slice under
   uniform random actions.

This notebook produces `data/processed/sigma_y_pretrain.json`, the only
artifact downstream notebooks (04+) read for prior elicitation.

In [ ]:
import json, sys
from pathlib import Path

NB_ROOT = Path.cwd().resolve()
SRC = NB_ROOT.parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from project import data, env, targets
from project.utils import PROCESSED_DIR, set_seed, ensure_dirs

ensure_dirs()
rng = set_seed()
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.spines.top": False, "axes.spines.right": False})

## Transaction cost: shape vs realised vol

In [ ]:
spy = data.load_spy()
train = spy.loc[data.TRAIN_START:data.TRAIN_END]

c_t = env.transaction_cost(train)
sigma_t = c_t.sub(env.C0_DEFAULT).div(env.LAMBDA_DEFAULT)  # back out sigma from cost

fig, ax = plt.subplots(figsize=(10, 3.0))
ax.plot(c_t.index, c_t * 1e4, color="C3", linewidth=0.6)
ax.axhline(env.C0_DEFAULT * 1e4, color="black", linestyle=":", linewidth=0.6, label=f"c_0 = {env.C0_DEFAULT*1e4:.0f} bps")
ax.set_ylabel("c_t (bps)")
ax.set_title("Per-day transaction cost on SPY training window")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

print(f"cost mean = {c_t.mean()*1e4:.2f} bps   max = {c_t.max()*1e4:.2f} bps   (c_0 = {env.C0_DEFAULT*1e4:.0f} bps)")

## Reward decomposition: long-only vs uniform-random actions

Two sanity examples on a 5-year slice:

- *Long-only* (`a_t = 1`, no trades after entry): `R_t = r_{t+1} - 0`. The
  resulting cumulative reward must equal SPY's log return over the window
  (modulo the initial entry cost).
- *Uniform random*: positive-mean trading cost drags the cumulative reward
  visibly below the long-only path; the gap is the cumulative cost paid.

In [ ]:
slice_ = spy.loc["2000":"2004"]
n = len(slice_)

actions_long = np.ones(n, dtype=int)
actions_rand = rng.choice([-1, 0, 1], size=n)

R_long = env.step_reward(slice_, actions_long, initial_position=1.0).fillna(0.0)
R_rand = env.step_reward(slice_, actions_rand, initial_position=0.0).fillna(0.0)

cum_long = R_long.cumsum()
cum_rand = R_rand.cumsum()

# Buy-and-hold benchmark: log return of SPY itself across the slice.
bh = np.log(slice_["Close"]).diff().fillna(0.0).cumsum()

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(bh.index, bh, color="black", linewidth=0.7, label="buy-and-hold log return")
ax.plot(cum_long.index, cum_long, color="C0", linewidth=0.7, linestyle="--", label="a_t = +1 (after one entry cost)")
ax.plot(cum_rand.index, cum_rand, color="C3", linewidth=0.7, label="a_t ~ Uniform({-1,0,+1})")
ax.set_ylabel("cumulative reward")
ax.set_title("Reward sanity check on SPY 2000-2004")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## Pre-training: estimate `sigma_y_hat` on ^GSPC 1990-1992

The hierarchical prior depends on `sigma_y_hat` (proposal section 2.4):
`tau_0 = 5 sigma_y_hat`, `Lambda_0 = sigma_y_hat^2 I_p`, `b_0 = sigma_y_hat^2`.
We freeze it once on the pre-training window and never touch it again.

Pre-training uses ^GSPC under **uniform random actions** -- `pi_b^(0)` per
proposal section 3 step 1. This gives an unbiased MC estimate of
`Q^{pi_b^{(0)}}` and hence a defensible scale for `y`.

In [ ]:
gspc = data.load_pretrain()
n_pre = len(gspc)
print(f"pretrain window: {gspc.index.min().date()} -> {gspc.index.max().date()}  ({n_pre} days)")

# Fix the actions used to compute sigma_y so the calibration is reproducible.
actions_pre = rng.choice([-1, 0, 1], size=n_pre)
R_pre = env.step_reward(gspc, actions_pre, initial_position=0.0)
y_pre = targets.mc_returns(R_pre)  # H=60, gamma=0.95 by default

sigma_y_hat = float(y_pre.std(ddof=1))
print(f"len(y_pre)    = {len(y_pre)}")
print(f"E[y_pre]      = {y_pre.mean():+.5f}")
print(f"sigma_y_hat   = {sigma_y_hat:.5f}")

artifact_path = PROCESSED_DIR / "sigma_y_pretrain.json"
artifact_path.write_text(json.dumps({
    "sigma_y_hat": sigma_y_hat,
    "ticker": data.PRETRAIN_SYMBOL,
    "start": data.PRETRAIN_START,
    "end": data.PRETRAIN_END,
    "n_days": n_pre,
    "n_targets": int(len(y_pre)),
    "behavior_policy": "uniform({-1,0,+1})",
    "H": targets.H_DEFAULT,
    "gamma": targets.GAMMA_DEFAULT,
}, indent=2))
print(f"wrote {artifact_path}")

## Per-action MC return distribution on the SPY training window

Under the *same* uniform-random behavior policy, compute `y_t` per
sampled action on SPY 1994-2019 and visualise the per-action
distributions. These are the regression targets that the Gibbs sampler
in notebook 04 will fit.

In [ ]:
actions_train = rng.choice([-1, 0, 1], size=len(train))
R_train = env.step_reward(train, actions_train, initial_position=0.0)
y_train = targets.mc_returns(R_train)

# Align actions to the y_t index (mc_returns drops rows where the window has NaN).
a_aligned = pd.Series(actions_train, index=train.index).loc[y_train.index]

print(f"len(y_train) = {len(y_train)}")
for a in env.ACTIONS:
    yi = y_train[a_aligned == a]
    print(f"  a={a:+d}: n={len(yi):>5}  mean={yi.mean():+.4f}  std={yi.std():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
colors = {-1: "C3", 0: "0.5", 1: "C0"}
for a in env.ACTIONS:
    yi = y_train[a_aligned == a]
    ax.hist(yi, bins=70, color=colors[a], alpha=0.55, density=True, label=f"a = {a:+d}  (n={len(yi)})")
ax.axvline(0, color="black", linewidth=0.5, linestyle=":")
ax.set_xlabel("MC return  y_t")
ax.set_ylabel("density")
ax.set_title("Per-action MC return on SPY 1994-2019, uniform-random behavior policy")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## Sanity assertions

In [ ]:
# Cost is non-negative and >= c_0 wherever defined
assert (c_t.dropna() >= env.C0_DEFAULT - 1e-12).all()

# Long-only with initial_position=1 has zero cost component (no flips after t=0)
R_long_no_init_cost = env.step_reward(slice_, actions_long, c0=env.C0_DEFAULT, lam=env.LAMBDA_DEFAULT, initial_position=1.0)
R_long_zero_cost   = env.step_reward(slice_, actions_long, c0=0.0,            lam=0.0,                initial_position=1.0)
np.testing.assert_allclose(R_long_no_init_cost.dropna().to_numpy(),
                           R_long_zero_cost.dropna().to_numpy(), atol=1e-12)

# sigma_y_hat is positive and finite
assert sigma_y_hat > 0 and np.isfinite(sigma_y_hat)

# y_t targets exist for all three actions on the training slice (under random policy)
for a in env.ACTIONS:
    assert (a_aligned == a).sum() > 0

print("OK")